In [ ]:
import uproot
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt
import mplhep

In [ ]:
events = uproot.open("/afs/cern.ch/work/a/amascell/FASERCal/FASERCal_converter/build/output_22517_00000.root:Events").arrays()
feb_vars = ['feb_id', 'channel_id', 'amplitude_lg', 'amplitude_hg', 
            'hit_time_rise', 'hit_time_fall', 'gts_time_rise', 'gts_time_fall']

In [ ]:
# np.unique(ak.flatten(events['feb_id']))

In [ ]:
def chunk_list(lst, size):
    return [lst[i:i + size] for i in range(0, len(lst), size)]

In [ ]:
# feb_events = events[feb_vars]
# np.unique(ak.flatten(feb_events['channel_id'][ak.num(feb_events['hit_time_rise'], axis=-1) > 0]))

In [ ]:
select_signal_withTime = True

### Plot Amplitude Distributions

In [ ]:
import pandas as pd
mplhep.style.use("CMS")

amp_var_dict = {
    'amplitude_lg': (200, [0, 200]),
    'amplitude_hg': (600, [0, 600])
}

for feb_id in np.unique(ak.flatten(events['feb_id'])):
    feb_events = events[feb_vars]
    feb_events = feb_events[feb_events['feb_id'] == feb_id]
    
            
    # For each event, check no channel_id appears more than once
    counts = ak.run_lengths(ak.sort(feb_events['channel_id'], axis=-1))
    assert not ak.any(ak.any(counts > 1, axis=-1)), \
        "More than 1 amplitude value per channel per event"

    if select_signal_withTime:
        feb_events = feb_events[ak.num(feb_events['hit_time_rise'], axis=-1) > 0]
        
    print(feb_events['hit_time_rise'])

    unique_channels = np.unique(ak.flatten(feb_events['channel_id']))

    for amp_var, (hist_bins, hist_range) in amp_var_dict.items():

        # Flatten once and build a DataFrame grouped by channel
        flat_channels = ak.to_numpy(ak.flatten(feb_events['channel_id'], axis=-1))
        flat_amplitudes = ak.to_numpy(ak.flatten(feb_events[amp_var], axis=-1))

        # Drop invalid amplitudes
        valid_mask = flat_amplitudes != -1
        flat_channels = flat_channels[valid_mask]
        flat_amplitudes = flat_amplitudes[valid_mask]

        # Group amplitudes by channel in one shot
        df = pd.DataFrame({'channel': flat_channels, 'amplitude': flat_amplitudes})
        grouped = df.groupby('channel')['amplitude']

        # Pre-compute all histograms before plotting
        histograms = {
            ch: np.histogram(group.values, bins=hist_bins, range=hist_range)
            for ch, group in grouped
        }

        for ch_list in chunk_list(list(unique_channels), 8):
            fig, ax = plt.subplots(figsize=(10, 7))

            for ch in ch_list:
                if ch not in histograms:
                    continue
                counts, bin_edges = histograms[ch]
                ax.stairs(counts, bin_edges, linewidth=1.5, label=f"Channel {ch}")

            ax.legend(loc="upper right", fontsize=20)
            ax.set_xlabel(amp_var)
            ax.set_ylabel("Entries")
            ax.set_yscale("log")
            plt.show()
            plt.close(fig)

### Study missing LG and HG values

In [ ]:
fig = plt.subplots(figsize=[10, 8])
mplhep.style.use("CMS")

for feb_id in np.unique(ak.flatten(events['feb_id'])):
    feb_events = events[feb_vars]
    feb_events = feb_events[feb_events['feb_id'] == feb_id]
    
    for amp_var in amp_var_dict:
    
        for ch in np.unique(ak.flatten(feb_events['channel_id'])):
            sel_events = (ak.sum((feb_events[amp_var] == -1) & (feb_events['channel_id'] == ch), axis=-1) > 0)
            # negative amplitude value for given channel id
            print(ch, sum(sel_events))
            if sum(sel_events) != 0:
                plt.hist(events['event_nbr_extended'][sel_events], histtype='step', linewidth=1.5, label=f"Channel {ch}")
        # plt.yscale('log')
        plt.legend()
        plt.ylabel("Entries")
        plt.xlabel("Event number")
        plt.show()
        plt.close()
# plt.savefig("Event_nbr_missing_hg.pdf")

In [ ]:
ch_nbrs = [7, 20, 22, 23, 29]
feb_events = events[feb_vars]
feb_events = feb_events[feb_events['feb_id'] == 0]

for ch_nbr in ch_nbrs:
    ch_events = feb_events[feb_events['channel_id'] == ch_nbr]
    ch_events_missHG = ch_events[ch_events['amplitude_hg'] == -1]
    amp_values = ak.flatten(ch_events_missHG['amplitude_lg'])
    amp_values_all = ak.flatten(ch_events['amplitude_lg'])
    plt.hist(amp_values, bins=200, range=[0, 200], histtype='step', linewidth=1.5, label="Missing HG value")
    plt.hist(amp_values_all, bins=200, range=[0, 200], histtype='step', linewidth=1.5, label="Total")
    plt.title(f"Channel {ch_nbr}")
    plt.yscale('log')
    plt.xlabel("amplitude_lg")
    plt.ylabel("Entries")
    plt.legend()
    plt.show()
    plt.close()